💡 **Environment:** `clamp-analyses`

# Description

**Per-disease performance metrics.** NB10/11 report performance **pooled** across all (drug, disease) pairs; the paired bootstrap in `../signif_test/` found no pairwise AUROC difference significant. Pooled AUROC also cannot separate real signal from **disease-popularity** confounds.

This notebook computes AUROC and AUPRC **within each disease** (across that disease's candidate drugs). Fixing the disease node removes disease-level popularity **by construction** — the rigorous version of "restrict to popular diseases."

It is a **read-only consumer** of `../signif_test/predictions_paired.pkl` (the already-aligned 685-pair × 4-method frame, produced by `signif_test/00_aggregate_predictions.ipynb`).

For each disease × method we record `n_pos`, `n_neg`, `n_total`, `base_rate = n_pos/n_total`, `auroc`, `auprc`, and the headline AUPRC summary **`auprc_log2_enrich = log2(auprc / base_rate)`** (fold-enrichment over the disease prior: 0 = no better than prior, +1 = 2× prior precision). AUROC/AUPRC are defined only for diseases with **≥1 positive and ≥1 negative** drug; `eligible_strict` flags the stricter **≥3 & ≥3** subset. Output: the long frame `per_disease_metrics.csv` (one row per `(trait, method)`), consumed by `01_per_disease_summary.ipynb`.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

from pyprojroot import here

# Settings

In [3]:
# The 4 methods, in a fixed order (matches ../signif_test). gene_based is the
# single-gene baseline (NB06); the three module_based_* are the LV models.
METHOD_ORDER = [
    'gene_based',
    'module_based_archs4',
    'module_based_gtex',
    'module_based_recount2',
]

# Eligibility for a per-disease AUROC/AUPRC: AUROC is undefined with a single
# class, so a disease needs >=1 positive AND >=1 negative drug. STRICT is the
# less-noisy subset used as a robustness anchor.
MIN_PER_CLASS = 1
MIN_PER_CLASS_STRICT = 3

INPUT_PKL = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/'
    'signif_test/predictions_paired.pkl')

OUTPUT_DIR = here(
    'output/03_model_biology/00_archs4/02_drug_disease_associations/per_disease_test')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
display(INPUT_PKL)
display(OUTPUT_DIR)

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/signif_test/predictions_paired.pkl')

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/per_disease_test')

# Load paired predictions

The aligned long frame from `../signif_test`: `[trait, drug, method, score, true_class]`, one row per (drug, disease, method). `trait` is the DOID disease id; the (trait, drug) pair universe is identical across the four methods (asserted below).

In [4]:
predictions_avg = pd.read_pickle(INPUT_PKL)
display(predictions_avg.shape)
display(predictions_avg.head())
display(predictions_avg['method'].value_counts())

(2740, 5)

,trait,drug,method,score,true_class
0,DOID:0050741,DB00215,gene_based,316134.3,1.0
1,DOID:0050741,DB00215,module_based_archs4,324870.1,1.0
2,DOID:0050741,DB00215,module_based_gtex,349350.5,1.0
3,DOID:0050741,DB00215,module_based_recount2,398655.9,1.0
4,DOID:0050741,DB00704,gene_based,387103.6,1.0


method
gene_based               685
module_based_archs4      685
module_based_gtex        685
module_based_recount2    685
Name: count, dtype: int64

In [5]:
# Coverage / pairing check: every method must cover the IDENTICAL set of
# (trait, drug) pairs, and labels must be consistent across methods. This is
# the same shared-universe guarantee ../signif_test relies on.
assert not predictions_avg.isna().any().any(), 'unexpected NaNs in predictions_paired.pkl'
missing = [m for m in METHOD_ORDER if m not in set(predictions_avg['method'].unique())]
assert not missing, f'methods absent from predictions_paired.pkl: {missing}'

pair_sets = {
    m: set(map(tuple, g[['trait', 'drug']].itertuples(index=False, name=None)))
    for m, g in predictions_avg.groupby('method', observed=True)
}
ref = pair_sets[METHOD_ORDER[0]]
for m in METHOD_ORDER[1:]:
    assert pair_sets[m] == ref, f'{m} covers a different (trait, drug) set than {METHOD_ORDER[0]}'

# Labels are a function of (trait, drug) only -> must agree across methods.
lbl = predictions_avg.groupby(['trait', 'drug'], observed=True)['true_class'].nunique()
assert (lbl == 1).all(), 'true_class disagrees across methods for some (trait, drug)'

n_pairs = len(ref)
n_diseases = predictions_avg['trait'].nunique()
display(f'{n_pairs} unique (drug, disease) pairs across {n_diseases} diseases, '
        f'{len(METHOD_ORDER)} methods')

'685 unique (drug, disease) pairs across 57 diseases, 4 methods'

# Per-disease metrics

For each `(method, trait)` group (a disease's candidate drugs under one method) compute the within-disease AUROC and AUPRC. AUROC/AUPRC/`auprc_log2_enrich` are set to `NaN` for non-eligible diseases (all-positive or all-negative), where they are undefined or trivial; `n_pos`/`n_neg`/`base_rate` are always recorded.

In [6]:
def _disease_metrics(g):
    y = g['true_class'].values.astype(int)
    s = g['score'].values
    n_total = len(y)
    n_pos = int(y.sum())
    n_neg = n_total - n_pos
    base_rate = n_pos / n_total if n_total else np.nan
    eligible = (n_pos >= MIN_PER_CLASS) and (n_neg >= MIN_PER_CLASS)
    if eligible:
        auroc = roc_auc_score(y, s)
        auprc = average_precision_score(y, s)
        # Fold-enrichment of AUPRC over the disease prior, in log2 space.
        # base_rate is strictly in (0, 1) for eligible diseases, so this is defined.
        auprc_log2_enrich = float(np.log2(auprc / base_rate))
    else:
        auroc = np.nan
        auprc = np.nan
        auprc_log2_enrich = np.nan
    return pd.Series({
        'n_pos': n_pos,
        'n_neg': n_neg,
        'n_total': n_total,
        'base_rate': base_rate,
        'auroc': auroc,
        'auprc': auprc,
        'auprc_log2_enrich': auprc_log2_enrich,
        'eligible': eligible,
        'eligible_strict': (n_pos >= MIN_PER_CLASS_STRICT) and (n_neg >= MIN_PER_CLASS_STRICT),
    })


per_disease = (
    predictions_avg
    .groupby(['method', 'trait'], observed=True)
    .apply(_disease_metrics, include_groups=False)
    .reset_index()
)

# Tidy dtypes (groupby.apply can return object columns).
per_disease['method'] = pd.Categorical(per_disease['method'], categories=METHOD_ORDER, ordered=True)
for c in ['n_pos', 'n_neg', 'n_total']:
    per_disease[c] = per_disease[c].astype(int)
for c in ['base_rate', 'auroc', 'auprc', 'auprc_log2_enrich']:
    per_disease[c] = per_disease[c].astype(float)
for c in ['eligible', 'eligible_strict']:
    per_disease[c] = per_disease[c].astype(bool)

per_disease = per_disease.sort_values(['method', 'trait']).reset_index(drop=True)
display(per_disease.shape)
display(per_disease.head(10))

(228, 11)

,method,trait,n_pos,n_neg,n_total,base_rate,auroc,auprc,auprc_log2_enrich,eligible,eligible_strict
0,gene_based,DOID:0050741,3,0,3,1.000000,NaN,NaN,NaN,False,False
1,gene_based,DOID:10283,16,7,23,0.695652,0.410714,0.717213,0.044035,True,True
2,gene_based,DOID:10534,9,0,9,1.000000,NaN,NaN,NaN,False,False
3,gene_based,DOID:10652,4,7,11,0.363636,0.178571,0.287338,-0.339749,True,True
4,gene_based,DOID:10763,58,27,85,0.682353,0.363346,0.600091,-0.185336,True,True
5,gene_based,DOID:11054,8,0,8,1.000000,NaN,NaN,NaN,False,False
6,gene_based,DOID:1115,8,1,9,0.888889,1.000000,1.000000,0.169925,True,False
7,gene_based,DOID:11476,8,4,12,0.666667,0.593750,0.779248,0.225117,True,True
8,gene_based,DOID:11714,1,0,1,1.000000,NaN,NaN,NaN,False,False
9,gene_based,DOID:12306,1,0,1,1.000000,NaN,NaN,NaN,False,False


# Eligibility summary (sanity)

In [7]:
# Eligibility is a property of (trait) only (same pair universe across methods),
# so the eligible-disease COUNT is identical for every method. Verify and report.
elig_per_method = per_disease.groupby('method', observed=True)['eligible'].sum()
strict_per_method = per_disease.groupby('method', observed=True)['eligible_strict'].sum()
display(pd.DataFrame({'n_eligible': elig_per_method, 'n_strict': strict_per_method}))

n_eligible = int(elig_per_method.iloc[0])
n_strict = int(strict_per_method.iloc[0])
assert (elig_per_method == n_eligible).all(), 'eligible count differs across methods (should not)'
assert (strict_per_method == n_strict).all(), 'strict count differs across methods (should not)'

# Diseases that are AUROC-undefined (excluded by construction): all-positive or
# all-negative. Report the split -- these are the most label-imbalanced diseases,
# so their exclusion is non-random (NEXT_STEPS Active #3).
one_method = per_disease[per_disease['method'] == METHOD_ORDER[0]]
n_all_pos = int(((one_method['n_neg'] == 0) & (one_method['n_pos'] > 0)).sum())
n_all_neg = int(((one_method['n_pos'] == 0) & (one_method['n_neg'] > 0)).sum())
print(f'Total diseases: {n_diseases}')
print(f'AUROC-eligible (>={MIN_PER_CLASS} pos & >={MIN_PER_CLASS} neg): {n_eligible}')
print(f'Strict (>={MIN_PER_CLASS_STRICT} pos & >={MIN_PER_CLASS_STRICT} neg): {n_strict}')
print(f'AUROC-undefined: {n_diseases - n_eligible} '
      f'({n_all_pos} all-positive, {n_all_neg} all-negative)')

# Quick macro-mean sanity vs the NEXT_STEPS preliminary (archs4 ~0.649, gene ~0.598).
macro = (per_disease[per_disease['eligible']]
         .groupby('method', observed=True)['auroc'].mean())
display(macro.rename('macro_mean_auroc'))

,n_eligible,n_strict
method,,
gene_based,33,17
module_based_archs4,33,17
module_based_gtex,33,17
module_based_recount2,33,17


Total diseases: 57
AUROC-eligible (>=1 pos & >=1 neg): 33
Strict (>=3 pos & >=3 neg): 17
AUROC-undefined: 24 (19 all-positive, 5 all-negative)


method
gene_based               0.657647
module_based_archs4      0.635574
module_based_gtex        0.640070
module_based_recount2    0.621982
Name: macro_mean_auroc, dtype: float64

# Save

In [8]:
out_csv = OUTPUT_DIR / 'per_disease_metrics.csv'
per_disease.to_csv(out_csv, index=False)
display(out_csv)
display(per_disease.shape)

PosixPath('/home/miltondp/projects/clamp/clamp-analyses/output/03_model_biology/00_archs4/02_drug_disease_associations/per_disease_test/per_disease_metrics.csv')

(228, 11)